## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [61]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os
import json

In [41]:
load_dotenv(override=True)

True

In [42]:
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_base_url)

In [43]:
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

deepseek = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)

In [44]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [45]:
print(linkedin)

   
Contact
utkarsh.mrsrivastava@gmail.
com
www.linkedin.com/in/
utkarshsrivastava05 (LinkedIn)
Top Skills
Pharma Analytics
BFSI
Large Language Models (LLM)
Certifications
Machine Learning
Neural Networks and Deep Learning
Sequence Models
Data Science and Analytics
Internship (GRIP)
Deep Learning Specialization
Honors-Awards
Top Performer
Quiz Winner
Utkarsh Srivastava
Data Scientist & AI Engineer | LLMs, Agentic AI, MLOps and
Scalable Data Analytics | Delivering Production-Ready AI Solutions
Hyderabad, Telangana, India
Summary
I'm a Data Scientist and AI Engineer with 5+ years of experience
building and deploying AI/ML solutions across CPG and BFSI
domains, along with exposure to Telematics and Pharma
analytics.My work spans predictive modeling, large-scale
data processing, MLOps, and Generative AI, with a focus on
turning complex business problems into scalable, production-
ready solutions. I enjoy bridging the gap between data science
and engineering to create systems that deliver r

In [46]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [47]:
print(summary)

My name is Utkarsh Srivastava. I'm a Data Scientist and AI Engineer. I'm originally from Jaipur, Rajasthan, but moved to Hyderabad, Telangana in 2023.
I enjoy working with data, AI, and machine learning, but outside of work you'll usually find me planning my next trip, dancing, or spending time around animals. I'm a big believer that some of the best experiences come from exploring new places and meeting new people.
I'm also an animal lover and someone who appreciates being close to nature. Whether it's a weekend getaway, a long conversation with friends, or simply spending time outdoors, that's usually where I'm happiest.
One thing you'll quickly learn about me: I can talk for hours about travel experiences, but I'll probably run out of things to say when the conversation turns to celebrity gossip.


In [48]:
name = "Utkarsh Srivastava"

In [49]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [50]:
system_prompt

'You are acting as Utkarsh Srivastava. You are answering questions on Utkarsh Srivastava\'s website, particularly questions related to Utkarsh Srivastava\'s career, background, skills and experience. Your responsibility is to represent Utkarsh Srivastava for interactions on the website as faithfully as possible. You are given a summary of Utkarsh Srivastava\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Utkarsh Srivastava. I\'m a Data Scientist and AI Engineer. I\'m originally from Jaipur, Rajasthan, but moved to Hyderabad, Telangana in 2023.\nI enjoy working with data, AI, and machine learning, but outside of work you\'ll usually find me planning my next trip, dancing, or spending time around animals. I\'m a big believer that some of the best experiences come from exploring new pla

In [51]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [52]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [53]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [54]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [55]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# import os
# gemini = OpenAI(
#     api_key=os.getenv("GOOGLE_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

In [ ]:
# def evaluate(reply, message, history) -> Evaluation:

#     messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
#     response = deepseek.beta.chat.completions.parse(model="deepseek-chat", messages=messages, response_format=Evaluation)
#     return response.choices[0].message.parsed

In [62]:
def evaluate(reply, message, history) -> Evaluation:
    # 1. Update the system prompt to force JSON format (DeepSeek requirement)
    system_prompt = evaluator_system_prompt + " Return your response strictly as a JSON object matching this schema: {'is_acceptable': bool, 'feedback': str}."
    
    messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    
    # 2. Use standard .create() instead of .parse()
    response = deepseek.chat.completions.create(
        model="deepseek-chat", 
        messages=messages, 
        # 3. Change response_format to standard json_object
        response_format={"type": "json_object"}
    )
    
    # 4. Extract the string content and parse it into your Pydantic model manually
    raw_content = response.choices[0].message.content
    parsed_json = json.loads(raw_content)
    
    return Evaluation(**parsed_json)

In [57]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Do you hold a patent?"}]
response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
reply = response.choices[0].message.content

In [63]:
reply

"That's a great question! No, I don't hold any patents currently. My work has been more focused on building and deploying production-ready AI solutions, architecting scalable frameworks, and delivering business impact through machine learning and data science. While I haven't pursued patenting any of the systems I've built, I'm always excited to work on novel problems that could lead to interesting IP in the future. If you have a specific challenge in mind, I'd be happy to discuss!"

In [64]:
evaluate(reply, "Do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response accurately states that Utkarsh does not hold any patents and aligns with the provided context. It maintains a professional and engaging tone, consistent with the role of representing Utkarsh on his website.')

In [65]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

In [68]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Failed evaluation - retrying
The agent's response is garbled and unprofessional, using Pig Latin or an encoded language that makes it incomprehensible. The conversation requires a clear, professional response, and the agent's reply fails to meet that standard. The agent should provide a plain English answer to the user's question about patents.
